# **Starting place for, do some imports and define some variables.**


*   Location
*   Staging_Bucket for Agent Engine Deployment
*   Project_ID

For Agent Engine Deployment
You will be prompted for your Google Maps API.  Subesequent runs will not request it.

In [2]:
# 1. Install required packages with the correct extras
!pip install --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk requests googlemaps

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://chal5_stage_bucket"

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET,)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

# 3. Securely prompt for your Google Maps API key
# If you already ran this, you can skip the input if the variable exists
try:
    if not os.environ.get("GOOGLE_MAPS_API_KEY"):
        maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
        os.environ["GOOGLE_MAPS_API_KEY"] = maps_key
except NameError:
    maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
    os.environ["GOOGLE_MAPS_API_KEY"] = maps_key

Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c
Enter your Google Maps API Key (starts with AIzaSy...): ··········


### **1. Weather Integration Tool**
This section defines the `get_us_weather_by_city_name` tool. It performs a two-step process:
1.  **Geocoding**: Uses the `googlemaps` library to convert a human-readable city name into Latitude and Longitude coordinates.
2.  **NWS API Call**: Uses the coordinates to query the National Weather Service (weather.gov) API to retrieve real-time regional forecasts.

In [3]:
#This is a combined function to get the users lat/long based on a US City and then provide weather for that location.
import os
import requests
import googlemaps
from typing import Any

def get_us_weather_by_city_name(city_name: str) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service
    by automatically converting a city/location name string into coordinates.
    """
    # 1. Fetch Coordinates from Google Maps
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)
    geocode_result = gmaps.geocode(city_name)

    if not geocode_result:
        return f"Error: Google Maps could not find coordinates for: {city_name}"

    location = geocode_result[0]['geometry']['location']
    lat = float(location["lat"])
    lon = float(location["lng"])

    # 2. Fetch Forecast from NWS
    headers = {'User-Agent': 'ADK-Weather-Agent'}
    # Ensure coordinates are formatted correctly in the URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return f"Error: {city_name} is outside NWS coverage jurisdiction (US locations only)."

    forecast_url = points_res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    forecast_summary = f"Weather Forecast for {city_name}:\n"
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

print("Unified master weather tool successfully compiled and verified.")

Unified master weather tool successfully compiled and verified.


Function used by Map Routing Agent

In [4]:
import googlemaps
from typing import Any

def get_route_directions(origin: str, destination: str) -> str:
    """
    Provides driving directions between two locations using Google Maps.
    Useful for evacuation and finding safe routes.
    """
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)

    try:
        # Request directions via driving mode
        directions_result = gmaps.directions(
            origin,
            destination,
            mode="driving",
            departure_time="now"
        )

        if not directions_result:
            return f"No routes found between {origin} and {destination}."

        route = directions_result[0]['legs'][0]
        summary = f"Safe Route from {origin} to {destination}:\n"
        summary += f"Estimated Distance: {route['distance']['text']}\n"
        summary += f"Estimated Duration: {route['duration']['text']}\n\n"
        summary += "Step-by-step instructions:\n"

        for i, step in enumerate(route['steps'], 1):
            # Clean HTML tags from instructions
            import re
            instruction = re.sub('<[^<]+?>', '', step['html_instructions'])
            summary += f"{i}. {instruction} ({step['distance']['text']})\n"

        return summary
    except Exception as e:
        return f"Error calculating route: {str(e)}"

### **2. Security Guardrails & Regulatory Logging**
To ensure safe and transparent AI behavior, we implement **ADK Callbacks**:
*   **Moderation Guardrail**: A `before_model` callback that scans user input for banned keywords (e.g., "hack"). If detected, the agent returns a security notice instead of processing the request.
*   **Audit Logging**: `log_user_prompt` and `log_model_response` callbacks record the conversation flow to the standard output for debugging and compliance.

In [5]:
# Callback functions for logging and content moderation.
import logging
import sys
import warnings
from typing import Optional
from google.adk.models import LlmRequest, LlmResponse

try:
    from google.adk.models import Content, Part
except ImportError:
    Content, Part = None, None

warnings.filterwarnings('ignore', category=UserWarning)

logger = logging.getLogger("ADK-FEMA-Lab")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

def log_user_prompt(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = getattr(parts[0], "text", "")
                logger.info("[%s] logging USER » %s", getattr(callback_context, "agent_name", "Default_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Logging error: {e}")
    return None

def moderation_guardrail(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    banned_keywords = ["hack", "exploit", "override", "bypass"]
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = str(getattr(parts[0], "text", "")).lower()
                if any(word in text for word in banned_keywords):
                    logger.warning("[%s] MODERATION TRIGGERED", getattr(callback_context, "agent_name", "Default_Bot"))
                    msg = "Security Notice: Banned term detected."
                    if Content and Part:
                        return LlmResponse(content=Content(role="model", parts=[Part(text=msg)]))
                    return LlmResponse(content={"role": "model", "parts": [{"text": msg}]})
    except Exception as e:
        logger.error(f"Guardrail error: {e}")
    return None

def log_model_response(callback_context, llm_response: LlmResponse) -> Optional[LlmResponse]:
    try:
        content = getattr(llm_response, "content", None)
        if content:
            parts = getattr(content, "parts", [])
            for part in parts:
                text = getattr(part, "text", None)
                if text:
                    logger.info("[%s] logging MODEL RESPONSE » %s", getattr(callback_context, "agent_name", "Weather_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Response logging error: {e}")
    return None

logger.info("Robust logger and guardrails ready.")

def combined_before_model_handler(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    guardrail_action = moderation_guardrail(callback_context, llm_request)
    if guardrail_action is not None: return guardrail_action
    return log_user_prompt(callback_context, llm_request)

2026-07-09 16:39:57,663 [INFO] Robust logger and guardrails ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
INFO:ADK-FEMA-Lab:Robust logger and guardrails ready.


### **3. Agent Orchestration & Native LoopAgent**
This section initializes the specialized agents and the main routing logic:


In [8]:
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, google_search_tool
from vertexai.preview import reasoning_engines

# 1. Specialized Sub-Agents
weather_agent = Agent(
    name="Weather_Agent",
    description="US weather forecast tool.",
    instruction="Provide current weather for the requested US city.",
    tools=[get_us_weather_by_city_name],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

google_search_agent = Agent(
    name="Google_Search_Agent",
    description="Web search tool.",
    instruction="Search for weather alerts and emergency advisories.",
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

map_route_agent = Agent(
    name="Map_Route_Agent",
    description="Navigation and routing tool.",
    instruction="Provide safe driving directions and routes for evacuation or reaching assistance centers.",
    tools=[get_route_directions],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)
# 2. Flattened Loop Components
# We provide the sub-agents directly to the LoopAgent logic rather than nesting them in an Orchestrator.
# This is to address resource/model exhaustion.
critique_agent = Agent(
    name="Critique_Agent",
    description="Verification agent.",
    instruction="Check the gathered findings for accuracy and safety.",
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

refiner_agent = Agent(
    name="Refiner_Agent",
    description="Final response agent.",
    instruction="Provide a professional final response based on verified findings.",
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# Update Clarity Agent to include the new routing capability
clarity_agent = LoopAgent(
    name="Clarity_Agent",
    description="Disaster response pipeline.",
    sub_agents=[weather_agent, google_search_agent, map_route_agent, critique_agent, refiner_agent],
    max_iterations=1
)

# Re-initialize the app with the updated Clarity Agent
root_agent = Agent(
    name='Main_Agent',
    description='App Entry Point',
    instruction="For weather, emergency alerts, or routing queries, use the Clarity_Agent.",
    tools=[agent_tool.AgentTool(agent=clarity_agent)]
)

app = reasoning_engines.AdkApp(agent=root_agent)
print('Architecture flattened: Orchestrator layer removed to prevent resource exhaustion.')

Architecture flattened: Orchestrator layer removed to prevent resource exhaustion.


Tests against local ADPApp.  This creates a function called run_agent_test that provides output in a formated way.

In [10]:
import time
from IPython.display import Markdown, display

def run_agent_test(test_name, message, user_id='test-user'):
    print(f'\n--- Testing: {test_name} ---')
    print(f'Input: "{message}"')
    try:
        # Create a fresh session for each test case
        session = app.create_session(user_id=user_id)
        s_id = session['id'] if isinstance(session, dict) else session.id

        # Fix: message must be passed as a keyword argument
        response_stream = app.stream_query(user_id=user_id, session_id=s_id, message=message)

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            display(Markdown(f'**Response:** {full_text}'))
        else:
            print('System: No text returned.')

        return full_text
    except Exception as e:
        print(f'Error: {e}')
        return None



In [ ]:
run_agent_test('Utah Fire', 'I am in Enoch, UT is there weather alert and what can I do?')

In [ ]:
# Test the new routing capability
run_agent_test('Evacuation Route', 'I am in Enoch, UT and need a safe route to the Cedar City high school.')